# 55. 累积分布图（ecdfplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 12 / 20 步：读懂连续变量的整体分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 核密度图（kdeplot）  →  **本章任务：** 累积分布图（ecdfplot）  →  **下一步：** 散点图（scatterplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

直接看一批数值时，直方图的分箱粗细会影响"大概多少"的判断，而 ECDF（经验累积分布函数）不需要任何分箱或带宽参数，就能从原始数据描出一条单调上升的累积曲线。



## 本章目标

学完本章，你将能够：

- **理解**：理解「累积分布图（ecdfplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「累积分布图（ecdfplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「累积分布图（ecdfplot）」并读出其中的结论。


## 55.1 适用场景

**背景引入**：直接看一批数值时，直方图的分箱粗细会影响"大概多少"的判断，而 ECDF（经验累积分布函数）不需要任何分箱或带宽参数，就能从原始数据描出一条单调上升的累积曲线。当你关心"客单价在多高时覆盖了 80% 的订单""不同品类的价格谁整体更高"这类问题，累计分布图能让你在任何阈值处直接读出覆盖比例、在任意比例处读出分位值，比直方图更适合做阈值覆盖率与分组比较。

打个比方：ecdfplot 像'沿一条路每隔一段插一根里程碑'——X 轴是数值，Y 轴是'到目前为止已经累积了多大比例的样本'。想在任意一个价位读出'覆盖了多少订单'，直接在对应 X 处往上找即可，不需要像直方图那样纠结箱子分多粗。它把'占比'这条曲线走得单调、平整、可直读。

比较分位数、阈值覆盖率或不同组的完整累计分布。


## 55.2 数据结构

一列连续数值，可按类别分组。


## 55.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 complementary=False 改为 complementary=True，对比累计分布与互补累计分布的曲线方向
2. 修改 stat="proportion" 为 stat="count"，观察比例与计数的纵轴差异
3. 在图上添加 axvline 标记特定分位数（如中位数位置），说明ECDF在分位数读取中的作用


## 55.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.ecdfplot()`、`ax.axhline()`、`ax.set()` | 比较分位数、阈值覆盖率或不同组的完整累计分布。 | 不理解阶梯线含义 |
| 进阶变体 | `plt.subplots()`、`sns.ecdfplot()`、`ax.axvline()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 组间样本量不等时比较count |
| 关键参数 | `stat` | proportion/count | 不理解阶梯线含义 |
| 关键参数 | `complementary` | 互补累计 | 组间样本量不等时比较count |
| 关键参数 | `hue` | 分组 | 把陡峭部分解释为时间变化 |
| 关键参数 | `weights` | 权重 | 不理解阶梯线含义 |


## 55.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-55 -->
### 数学推导｜经验累积分布函数

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜对给定阈值逐个判断。** $I_i(x)=\mathbf{1}(x_i\le x)$。

**第 2 步｜把满足条件的个数除以样本量。**

$$
\hat F_n(x)=\frac{\sum_iI_i(x)}{n}
$$

**第 3 步｜理解阶梯。** 每经过一个样本点，累计比例增加 $1/n$；因此经验中位数可写成满足 $\hat F_n(x)\ge0.5$ 的最小 $x$。

**把上面的关系收束为本章计算式：**

$$
\hat{F}_n(x)=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(x_i\le x)
$$

**符号解释：** $\hat{F}_n(x)$ 表示样本中不大于 $x$ 的比例。

**代码对应：** ECDF 的纵轴可直接解释为累计比例，适合比较中位数、尾部和阈值覆盖率。

**使用边界：** 曲线差异是描述性证据；组间样本量和抽样方式仍需报告。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 55.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(data=orders, x="order_value", color="#1a73e8", ax=ax)
ax.axhline(0.5, color="#9aa0a6", linestyle="--")
ax.set(title="订单金额累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 55.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


**练一练**：上面的基础图表用默认的"累计比例"画出了订单金额的累计分布，曲线一路爬升到 1。请以它为蓝本，把 `sns.ecdfplot(data=orders, x="order_value", ...)` 中的 `complementary` 参数补成 `True`，重画一版互补累计分布，观察曲线方向如何从"≤某值的比例"反转为">某值的比例"；再想一想：当客单价很小、很大时，互补曲线的取值应该分别接近多少？


In [ ]:
# 请在下方填写代码
# 任务：把下方 ecdfplot 的 complementary 填空（___）改为 True，重画订单金额的互补累计分布。
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(
    data=orders,
    x="order_value",
    complementary=___,  # 请把 ___ 填写为 True
    ax=ax,
)
ax.set(title="订单金额互补累计分布", xlabel="客单价（元）", ylabel="互补累计比例")
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(data=orders, x="order_value", complementary=True, ax=ax)
ax.set(title="订单金额互补累计分布", xlabel="客单价（元）", ylabel="互补累计比例")
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(
    data=orders, x="order_value", hue="category", palette="colorblind", ax=ax
)
ax.axvline(300, color="#d93025", linestyle="--", label="300元阈值")
ax.set(title="品类客单价累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 55.8 参数说明

- stat：proportion/count
- complementary：互补累计
- hue：分组
- weights：权重


## 55.9 结果解读

在任意X值读取累计比例，或在给定比例处读取分位值。


## 55.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 55.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 55.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 55.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 55.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 55.12 易错点提醒

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


## 55.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 55.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：按渠道分层累积分布，比较不同渠道的达标速度
# 【目标】累积分布能看出「达到某值已经覆盖多少比例」，分层后可直接对比渠道。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="channel"，看不同渠道累积比例的差异。
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(data=orders, x="order_value", hue="channel", ax=ax)
ax.set(title="分渠道累计分布", xlabel="客单价（元）", ylabel="累计比例")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：哪个渠道更快达到 50% 累计 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(
    data=marketing,
    x="conversion",
    hue="channel",
    complementary=True,
    palette="colorblind",
    ax=ax,
)
ax.set(title="转化率超过阈值的比例", xlabel="转化率阈值", ylabel="超过阈值的比例")
fig.tight_layout()
plt.show()


## 55.15 小结

用ECDF直接展示小于等于某值的样本比例，无需选择分箱或带宽。


### 55.15.1 你已经掌握

- 判断累积分布图（ecdfplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 55.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `stat` | proportion/count |
| `complementary` | 互补累计 |
| `hue` | 分组 |
| `weights` | 权重 |


### 55.15.3 需要注意

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


### 55.15.4 完成检查

- [ ] 能判断什么问题适合使用累积分布图（ecdfplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 55.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
